# SAQ method-stage approximation experiments

Heuristic-vs-optimal quality for each stage of the method (professor's experiments):
1. **Codebook** — cumsum-kmeans vs DP-optimal per-dim codebook
2. **Allocation** — greedy vs optimal bit allocation (+ analytic Bennett)
3. **Packing** — FFD vs exact-optimal byte packing

Point `RESULTS_BASE` at a local `results/` dir or a PACE-pulled results dir
(must contain `approx_codebook/`, `approx_alloc/`, `approx_packing/`).

In [ ]:
import os, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_BASE = Path(os.environ.get('VQ_RESULTS_DIR', REPO / 'results'))
DATASET = os.environ.get('VQ_DATASET', 'dbpedia_100k')
FIGS = RESULTS_BASE / '_figs'
print('RESULTS_BASE =', RESULTS_BASE)

# (Re)generate the figures from the CSVs.
subprocess.run([sys.executable, str(REPO / 'results' / 'plot_approx.py'),
                '--base', str(RESULTS_BASE), '--out', str(FIGS), '--dataset', DATASET], check=True)

## Headline — every stage vs optimal

In [ ]:
display(Image(str(FIGS / 'approx_combined.png')))

## Exp 1 — codebook (cumsum-kmeans vs DP-optimal)
`ratio = lloyd_mse / dp_mse` (1.0 = optimal). Near-optimal at low bpd, gap opens at high bpd.

In [ ]:
display(pd.read_csv(RESULTS_BASE / 'approx_codebook' / 'approx_codebook_summary.csv').round(4))
display(Image(str(FIGS / 'exp1_codebook.png')))

## Exp 2 — bit allocation (greedy vs optimal, + Bennett)
Greedy is exactly optimal (convex costs); the analytic Bennett model is far from optimal at mid bpd.

In [ ]:
display(pd.read_csv(RESULTS_BASE / 'approx_alloc' / 'approx_alloc.csv').round(5))
display(Image(str(FIGS / 'exp2_alloc.png')))

## Exp 3 — byte packing (FFD vs exact optimal)
FFD == OPT throughout; the gap to the perfect-packing LB is intrinsic (large codes can't fill a byte).

In [ ]:
display(pd.read_csv(RESULTS_BASE / 'approx_packing' / 'approx_packing.csv').round(4))
display(Image(str(FIGS / 'exp3_packing.png')))

## Recall benchmark (method comparison)
If a recall run is present, load its CSV + Pareto plot. Set `RECALL_DIR` to the
recall results dir (has `results_*.csv` / `pareto_*.png` or `results_checkpoint.csv`).

In [ ]:
RECALL_DIR = Path(os.environ.get('VQ_RECALL_DIR', RESULTS_BASE / 'recall'))
csvs = sorted(RECALL_DIR.glob('results*.csv')) if RECALL_DIR.exists() else []
if csvs:
    df = pd.read_csv(csvs[-1])
    print('methods:', sorted(df['method'].unique()))
    # recall@10 vs compression pivot
    display(df.pivot_table(index='bpd', columns='method', values='recall_at_10').round(4))
    for png in sorted(RECALL_DIR.glob('pareto*.png')):
        display(Image(str(png)))
else:
    print('No recall results found at', RECALL_DIR, '- run the recall benchmark first.')